# Logistic regression as a single neuron

**Lecture 11 · Notebook 00 · CMOR 438 / INDE 577**  
**Core:** about 230 minutes · **Practice and extension:** about 70 minutes

This laboratory begins with the same affine score used by linear regression,

$$
z_i=b+\boldsymbol w^T\boldsymbol x_i,
$$

but asks a different scientific question: how should evidence in
$\boldsymbol x_i$ change the modeled probability of a binary event
$Y_i\in\{0,1\}$? We will build the resulting model as one computational
neuron, derive its loss and gradient, verify the implementation, and compare it
with a trusted library estimator. We then begin the professional evaluation workflow: baselines, confusion matrices, probability quality, ranking, thresholds, uncertainty, and population context.

The worked example uses measurements derived from digitized images of breast
masses. It is an instructional model of a public dataset—not a clinical device,
diagnostic claim, or substitute for domain validation.

## How to use this notebook

1. Read each derivation before running the cell that implements it.
2. Predict the sign or shape of a result at every **checkpoint**.
3. Keep the final test partition sealed until the section that explicitly opens it.
4. Distinguish four objects throughout:

   $$
   \text{affine score}\longrightarrow\text{probability model}
   \longrightarrow\text{training objective}\longrightarrow\text{decision policy}.
   $$

5. Restart the **Rice DSM** kernel and run all cells before treating the notebook
   as completion evidence.

The notebook is an experimental laboratory. Reusable model code ultimately
belongs in `src/rice_dsm/`, behavioral checks in `tests/`, and trained artifacts
in a versioned model workflow.


## Learning objectives

By the end, you should be able to:

- distinguish a logit, an estimated probability, and a thresholded decision;
- derive binary cross-entropy from a conditional Bernoulli likelihood;
- prove that the per-example logit derivative is $p_i-y_i$;
- explain precisely how logistic and linear regression are alike and different;
- implement a numerically stable logistic neuron and verify its gradient;
- interpret the Hessian $X^TWX/n$ and the role of feature scaling;
- preserve training, validation, and test boundaries in a classification workflow; and
- compare a transparent implementation with a regularized library pipeline;
- derive sensitivity, specificity, precision, and negative predictive value from a confusion matrix; and
- explain how thresholds, prevalence, sampling, uncertainty, and deployment population change performance claims.

## Why this matters in industry and science

Binary outcomes appear in equipment failure, fraud review, quality control,
ecology, remote sensing, and medical research. A production system rarely needs
only a class label. It may need a ranked queue, an estimated event probability,
an abstention rule, or a capacity-constrained intervention policy.

Logistic regression remains valuable because its assumptions and failure modes
are visible. It provides a baseline for larger neural networks, a compact model
for limited data, and a direct bridge from likelihood to gradient-based
learning. But a number between zero and one is not automatically calibrated,
causal, fair, or safe. Those are evidence claims, not consequences of the
sigmoid function.


## Historical context: a growth curve became a statistical link

The S-shaped logistic curve predates machine learning. Verhulst used logistic
growth in nineteenth-century population modeling. Berkson's 1944 bio-assay
paper developed the logistic function for binary response analysis, and Cox's
1958 work treated regression for binary sequences. Modern software often calls
the fitted object `LogisticRegression`, although its response is a probability
and its training criterion is a likelihood—not squared-error regression.

The neural-network language came later. A logistic regression model can be
drawn as a single neuron because it computes an affine score followed by a
sigmoid activation. That representation is useful only if we keep the
statistical contract visible.


In [ ]:
from __future__ import annotations

# Standard-library tools make database access and fitted results explicit.
import sqlite3
from dataclasses import dataclass

# Arrays, tables, and plots expose each mathematical object directly.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Trusted estimators and metrics provide independent reference behavior.
from sklearn.calibration import CalibrationDisplay
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    brier_score_loss,
    confusion_matrix,
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import rice_dsm
from rice_dsm.paths import course_database_path

RANDOM_SEED = 438577
assert rice_dsm.__version__
print(f"Rice DSM {rice_dsm.__version__}; split seed = {RANDOM_SEED}")


## Worked example: an image-derived binary outcome

The Wisconsin Diagnostic Breast Cancer dataset contains 569 observations. Each
row summarizes geometric and texture properties computed from a digitized
image of a fine-needle aspirate. The source target uses `0` for malignant and
`1` for benign. For this lesson we create the explicit event

$$
Y_i=\mathbf 1\{\text{diagnosis is malignant}\},
$$

so that $Y_i=1$ always names the event whose modeled probability we report.
This recoding is not cosmetic: every probability, confusion matrix, and policy
must name its positive class.

Before fitting, we will inspect the database catalog, column roles, row
identity, missingness, and class prevalence. Data discovery is part of the
modeling argument.


In [ ]:
# Resolve the repository database without depending on the notebook directory.
database_path = course_database_path()
database_uri = f"{database_path.as_uri()}?mode=ro"
course_database = sqlite3.connect(database_uri, uri=True)

# Ask SQLite what exists before relying on a remembered table name.
available_tables = pd.read_sql_query(
    """
    SELECT name AS table_name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
    """,
    course_database,
)
# Read provenance and task semantics from the catalog, not from memory.
dataset_record = pd.read_sql_query(
    """
    SELECT dataset_id, title, domain, task, observational_unit,
           target_column, target_description, row_count, source_url, source_note
    FROM dataset_catalog
    WHERE dataset_id = 'breast_cancer_wisconsin'
    """,
    course_database,
)
# The column dictionary separates identifiers, features, and target fields.
column_roles = pd.read_sql_query(
    """
    SELECT ordinal_position, column_name, role
    FROM dataset_columns
    WHERE dataset_id = 'breast_cancer_wisconsin'
    ORDER BY ordinal_position
    """,
    course_database,
)

assert "breast_cancer_observations" in set(available_tables["table_name"])
assert len(dataset_record) == 1
assert dataset_record.loc[0, "row_count"] == 569
dataset_record.T


In [ ]:
# Query only versioned repository data; no network call occurs in this notebook.
observations = pd.read_sql_query(
    """
    SELECT observation_id, mean_radius, mean_texture, mean_smoothness,
           mean_concave_points, worst_radius, worst_concave_points,
           diagnosis_code, diagnosis_label
    FROM breast_cancer_observations
    ORDER BY observation_id
    """,
    course_database,
    index_col="observation_id",
)
course_database.close()

# Recode the event so probability column 1 has an unambiguous meaning.
observations["malignant_event"] = (
    observations["diagnosis_label"] == "malignant"
).astype(int)

# Structural assertions turn assumptions about the source into executable checks.
assert observations.shape == (569, 9)
assert observations.index.is_unique
assert observations.isna().sum().sum() == 0
assert set(observations["diagnosis_label"]) == {"benign", "malignant"}
assert (observations["malignant_event"] == 1 - observations["diagnosis_code"]).all()

print(observations.head())
print("\nClass counts:\n", observations["diagnosis_label"].value_counts())


## Split roles before target-guided exploration

We construct three disjoint evidence sets:

- $\mathcal D_{\mathrm{train}}$: estimates scaler statistics and model parameters;
- $\mathcal D_{\mathrm{validation}}$: checks implementation and modeling choices;
- $\mathcal D_{\mathrm{test}}$: supports one final, predeclared performance report.

`stratify` preserves class proportions approximately. It does **not** solve
group leakage, temporal leakage, site shift, or duplicate-patient leakage. This
table does not expose a patient grouping or collection time, so a random split
is a limitation of the available schema, not a universal recommendation.


In [ ]:
# Reserve the final test rows before any target-guided visualization.
development_rows, test_rows = train_test_split(
    observations.index,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=observations["malignant_event"],
)

# Reserve validation rows from the remaining development population.
train_rows, validation_rows = train_test_split(
    development_rows,
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=observations.loc[development_rows, "malignant_event"],
)

# Copies make the three evidence roles explicit and prevent accidental mutation.
train_data = observations.loc[train_rows].copy()
validation_data = observations.loc[validation_rows].copy()
test_data = observations.loc[test_rows].copy()

assert set(train_data.index).isdisjoint(validation_data.index)
assert set(train_data.index).isdisjoint(test_data.index)
assert set(validation_data.index).isdisjoint(test_data.index)
assert len(train_data) + len(validation_data) + len(test_data) == len(observations)

split_summary = pd.DataFrame(
    {
        "rows": [len(train_data), len(validation_data), len(test_data)],
        "event_rate": [
            frame["malignant_event"].mean()
            for frame in (train_data, validation_data, test_data)
        ],
    },
    index=["train", "validation", "test"],
)
split_summary


## Inspect the training population before modeling

For the derivation we use one feature, `worst_concave_points`: the mean of the
three largest measurements of concave portions of the nucleus contour. One
feature lets us see the fitted curve. It does not make that variable causal,
sufficient, or suitable for clinical use.

The plot asks a named question: **do the class-conditional training
distributions overlap?** If they do, a hard separator cannot classify every
row correctly, and a probability model is more informative than pretending the
boundary is certain.


In [ ]:
one_feature = "worst_concave_points"

# Summaries expose range, spread, and class imbalance before optimization.
feature_summary = train_data.groupby("diagnosis_label")[one_feature].agg(
    ["count", "mean", "std", "min", "median", "max"]
)
print(feature_summary)

# Histograms use identical bins so visual area is comparable across classes.
feature_bins = np.linspace(
    train_data[one_feature].min(), train_data[one_feature].max(), 24
)
# Plot only training rows; test labels remain sealed.
fig, ax = plt.subplots(figsize=(9, 4.8))
for label, color in (("benign", "#1f77b4"), ("malignant", "#c44e52")):
    values = train_data.loc[train_data["diagnosis_label"] == label, one_feature]
    ax.hist(
        values,
        bins=feature_bins,
        alpha=0.55,
        density=True,
        label=label,
        color=color,
    )
ax.set(
    xlabel="worst concave points (source measurement units)",
    ylabel="density",
    title="Training distributions overlap: separation is not certainty",
)
ax.legend()
plt.show()

assert train_data[one_feature].nunique() > 100
assert 0.0 < train_data["malignant_event"].mean() < 1.0


## 1. Why not use the linear neuron unchanged?

Linear regression with one input computes

$$
\widehat y_i=b+w x_i
$$

and minimizes squared error. If $y_i\in\{0,1\}$, this *linear probability
model* can be useful for some analyses, but its fitted values are not constrained
to $[0,1]$, its variance assumptions do not match a Bernoulli response, and a
constant slope cannot represent saturation near probabilities zero and one.

**Checkpoint:** before running the next cell, sketch what happens when the
fitted line is evaluated beyond the observed feature range.


In [ ]:
# Fit only on training data; this model is a contrast, not the final estimator.
linear_probability_model = LinearRegression().fit(
    train_data[[one_feature]], train_data["malignant_event"]
)
feature_domain = np.linspace(
    -0.04, train_data[one_feature].max() * 1.15, 300
).reshape(-1, 1)
linear_probability = linear_probability_model.predict(feature_domain)

# Jittered binary labels reveal overlap without pretending labels are continuous.
rng = np.random.default_rng(RANDOM_SEED)
jitter = rng.normal(0.0, 0.018, size=len(train_data))
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.scatter(
    train_data[one_feature],
    train_data["malignant_event"] + jitter,
    alpha=0.35,
    s=22,
    color="#4c566a",
    label="training labels (jittered)",
)
# Extend the fitted affine response beyond the observed range to expose its domain.
ax.plot(feature_domain[:, 0], linear_probability, color="#c44e52", linewidth=2.5)
ax.axhline(0, color="black", linewidth=0.8)
ax.axhline(1, color="black", linewidth=0.8)
ax.set(
    xlabel="worst concave points",
    ylabel="event label or fitted value",
    title="An affine response is not constrained to the probability interval",
)
plt.show()

assert linear_probability.min() < 0 or linear_probability.max() > 1


## 2. The logistic single neuron

For input $\boldsymbol x_i\in\mathbb R^d$, weights
$\boldsymbol w\in\mathbb R^d$, and bias $b\in\mathbb R$, define

$$
z_i=b+\boldsymbol w^T\boldsymbol x_i,
\qquad
p_i=\sigma(z_i)=\frac{1}{1+e^{-z_i}}.
$$

The score $z_i$ is the **preactivation** and, in this model, also the
**logit**. Neural-network notation calls the value after the activation
$a_i$; probability notation calls it $p_i$. For logistic regression these
are the same numerical output,

$$
a_i=\sigma(z_i)=p_i=P(Y_i=1\mid\boldsymbol X_i=\boldsymbol x_i),
$$

but $p_i$ is the more informative name in this lesson because it preserves
the output's statistical meaning. This is a one-neuron computation:

```text
features x  ->  affine score z = b + w^T x  ->  sigmoid σ(z)  ->  p
```

The same affine score appears in linear regression. Changing the activation
changes the function class, and changing the likelihood changes the objective.
The complete mathematical specification is therefore:

| Component | Logistic single-neuron choice |
|---|---|
| target | $y_i\in\{0,1\}$ |
| preactivation | $z_i=\boldsymbol w^T\boldsymbol x_i+b$ |
| postactivation | $a_i=\sigma(z_i)=p_i$ |
| output meaning | $p_i=P(Y_i=1\mid\boldsymbol X_i=\boldsymbol x_i)$ |
| training loss | Bernoulli negative log-likelihood |
| score derivative | $\delta_i=\partial\ell_i/\partial z_i=p_i-y_i$ |
| decision | a separate threshold or cost policy |

![Forward and backward contract of one logistic neuron](../../textbook/figures/generated/logistic-single-neuron.png)

> **Why keep the entire specification visible?** The affine calculation by
> itself does not define an estimator. Its target domain, activation, loss,
> output meaning, and decision policy must agree.


In [ ]:
def sigmoid(logits: np.ndarray | float) -> np.ndarray:
    """Evaluate the logistic sigmoid without exponential overflow.

    Parameters
    ----------
    logits : array-like
        Finite affine scores of any shape.

    Returns
    -------
    numpy.ndarray
        Probabilities with the same shape as ``logits``.
    """

    # Normalize input so one implementation handles scalars and arrays alike.
    values = np.asarray(logits, dtype=float)
    probabilities = np.empty_like(values)
    nonnegative = values >= 0

    # For nonnegative z, exp(-z) is bounded by one.
    probabilities[nonnegative] = 1.0 / (
        1.0 + np.exp(-values[nonnegative])
    )

    # For negative z, rewrite sigmoid(z) using exp(z), which is also bounded.
    exp_values = np.exp(values[~nonnegative])
    probabilities[~nonnegative] = exp_values / (1.0 + exp_values)
    return probabilities


logit_domain = np.linspace(-10, 10, 500)
fig, ax = plt.subplots(figsize=(8.5, 4.6))
ax.plot(logit_domain, sigmoid(logit_domain), linewidth=2.5, color="#5e3c99")
ax.axvline(0, color="black", linewidth=0.8)
ax.axhline(0.5, color="black", linewidth=0.8, linestyle="--")
ax.set(
    xlabel="logit z",
    ylabel="σ(z)",
    title="The sigmoid maps every finite score into (0, 1)",
)
plt.show()

assert np.allclose(sigmoid(np.array([-1000.0, 0.0, 1000.0])), [0.0, 0.5, 1.0])


### How the sigmoid bends an affine score into a probability curve

The sigmoid has four properties worth seeing rather than memorizing:

$$
\lim_{z\to-\infty}\sigma(z)=0,
\qquad
\sigma(0)=\frac12,
\qquad
\lim_{z\to\infty}\sigma(z)=1,
\qquad
\sigma'(z)=\sigma(z)(1-\sigma(z)).
$$

For one raw feature $x$, write the logit as $z=b+wx$. When $w\ne0$,

$$
p(x)=\sigma(b+wx)
=\sigma\!\left(w(x-x_{0.5})\right),
\qquad
x_{0.5}=-\frac bw.
$$

The midpoint $x_{0.5}$ is where the curve crosses probability $0.5$. The sign
of $w$ chooses whether the curve rises or falls, and $|w|$ controls steepness.
The raw-feature slope is largest at the midpoint:

$$
\left.\frac{dp}{dx}\right|_{x=x_{0.5}}=\frac w4.
$$

Thus $b$ and $w$ do not independently mean “position” and “steepness” in raw
coordinates—the midpoint depends on their ratio. Scaling $x$ changes the
numerical value of $w$ and therefore must be stored with the model.


In [ ]:
# Parameterize candidate curves by an interpretable midpoint and logit slope.
raw_feature_domain = np.linspace(
    train_data[one_feature].min(), train_data[one_feature].max(), 400
)
feature_midpoint = float(train_data[one_feature].median())
shifted_midpoint = float(train_data[one_feature].quantile(0.65))
candidate_curves = [
    ("gentle, median midpoint", 25.0, feature_midpoint, "#1f77b4"),
    ("steep, median midpoint", 70.0, feature_midpoint, "#7b3294"),
    ("steep, shifted midpoint", 70.0, shifted_midpoint, "#c44e52"),
]

# The left panel connects the activation to its derivative and saturation.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
activation = sigmoid(logit_domain)
activation_derivative = activation * (1.0 - activation)
axes[0].plot(logit_domain, activation, linewidth=2.5, label="σ(z)")
axes[0].plot(
    logit_domain,
    activation_derivative,
    linewidth=2.2,
    label="σ′(z)=σ(z)(1-σ(z))",
)
axes[0].scatter([0], [0.5], color="black", zorder=3, label="midpoint")
axes[0].set(
    xlabel="affine score z",
    ylabel="function value",
    title="Sigmoid anatomy: midpoint, slope, and saturation",
)
axes[0].legend()

# The right panel maps candidate sigmoid curves onto the observed binary data.
plot_rng = np.random.default_rng(RANDOM_SEED + 12)
axes[1].scatter(
    train_data[one_feature],
    train_data["malignant_event"].to_numpy()
    + plot_rng.normal(0.0, 0.018, len(train_data)),
    alpha=0.28,
    s=20,
    color="#4c566a",
    label="training outcomes (jittered)",
)
for label, logit_slope, midpoint, color in candidate_curves:
    candidate_probability = sigmoid(
        logit_slope * (raw_feature_domain - midpoint)
    )
    axes[1].plot(
        raw_feature_domain,
        candidate_probability,
        linewidth=2.2,
        color=color,
        label=label,
    )
axes[1].set(
    xlabel="worst concave points",
    ylabel="candidate P(malignant event | measurement)",
    title="Many sigmoid curves can be mapped onto the same data",
)
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

# Every midpoint parameter must map to probability one half.
for _, logit_slope, midpoint, _ in candidate_curves:
    assert np.isclose(sigmoid(logit_slope * (midpoint - midpoint)), 0.5)


### Log-odds are linear, probabilities are not

Because $p=\sigma(z)$,

$$
\frac{p}{1-p}=e^z,
\qquad
\log\frac{p}{1-p}=z=b+\boldsymbol w^T\boldsymbol x.
$$

Thus a one-unit increase in $x_j$ adds $w_j$ to the log-odds and multiplies the
odds by $e^{w_j}$, holding other inputs fixed. It does **not** add a constant
amount to the probability; the probability change depends on the starting
logit. Association is not causation, and correlated inputs complicate any
isolated coefficient story.


In [ ]:
score_examples = np.array([-4.0, -2.0, 0.0, 2.0, 4.0])
probability_examples = sigmoid(score_examples)
odds_examples = probability_examples / (1.0 - probability_examples)

score_table = pd.DataFrame(
    {
        "logit_z": score_examples,
        "probability": probability_examples,
        "odds": odds_examples,
        "recovered_log_odds": np.log(odds_examples),
    }
)
assert np.allclose(score_table["recovered_log_odds"], score_examples)
score_table


### A probability is a parameter of a conditional distribution

Students often encounter three different objects and call all of them “the
probability.” Keep them separate:

- $Y_i$ is a random variable before its outcome is observed;
- $y_i\in\{0,1\}$ is the observed binary outcome for one unit; and
- $p_i=p(\boldsymbol x_i)$ is a model parameter for the distribution of
  possible outcomes among comparable units with information $\boldsymbol x_i$.

The logistic specification is the complete conditional Bernoulli distribution

$$
P(Y_i=y\mid\boldsymbol X_i=\boldsymbol x_i)
=p_i^y(1-p_i)^{1-y},\qquad y\in\{0,1\}.
$$

Its conditional mean and variance are

$$
E[Y_i\mid\boldsymbol X_i=\boldsymbol x_i]=p_i,
\qquad
\operatorname{Var}(Y_i\mid\boldsymbol X_i=\boldsymbol x_i)=p_i(1-p_i).
$$

A probability of $0.8$ is not a fractional label and does not assert that one
particular outcome is “80% malignant.” It says that, under the model and its
population assumptions, the event frequency would be about $80\%$ across many
comparable future units. The next individual outcome is still zero or one.

Probability also does not measure confidence that the model itself is correct.
Sampling uncertainty, parameter uncertainty, dataset shift, and model
misspecification are additional sources of uncertainty.


In [ ]:
# Use an independent seed so this probability experiment is reproducible.
probability_rng = np.random.default_rng(RANDOM_SEED + 11)
probability_grid = np.linspace(0.0, 1.0, 400)
repeated_probability = 0.70
repeated_outcomes = probability_rng.binomial(
    n=1, p=repeated_probability, size=500
)
bernoulli_running_mean = np.cumsum(repeated_outcomes) / np.arange(1, 501)

# Panel 1 shows that a Bernoulli distribution has mass only at zero and one.
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
bar_width = 0.22
for offset, event_probability in enumerate((0.2, 0.5, 0.8)):
    positions = np.array([0.0, 1.0]) + (offset - 1) * bar_width
    masses = np.array([1.0 - event_probability, event_probability])
    axes[0].bar(
        positions,
        masses,
        width=bar_width,
        alpha=0.75,
        label=f"p={event_probability:.1f}",
    )
axes[0].set(
    xticks=[0, 1],
    xlabel="possible outcome y",
    ylabel="probability mass",
    title="Bernoulli outcomes remain binary",
)
axes[0].legend()

# Panel 2 separates the conditional mean p from outcome variance p(1-p).
axes[1].plot(probability_grid, probability_grid, label="mean = p", linewidth=2.2)
axes[1].plot(
    probability_grid,
    probability_grid * (1.0 - probability_grid),
    label="variance = p(1-p)",
    linewidth=2.2,
)
axes[1].set(
    xlabel="Bernoulli parameter p",
    ylabel="conditional moment",
    title="Uncertainty is greatest near p=0.5",
)
axes[1].legend()

# Panel 3 illustrates frequency convergence without claiming every prefix is close.
axes[2].plot(np.arange(1, 501), bernoulli_running_mean, color="#6a3d9a")
axes[2].axhline(
    repeated_probability,
    color="black",
    linestyle="--",
    label="generating p=0.70",
)
axes[2].set(
    xlabel="number of comparable outcomes",
    ylabel="running event fraction",
    title="Probability becomes visible through repetition",
)
axes[2].legend()
plt.tight_layout()
plt.show()

assert set(np.unique(repeated_outcomes)) <= {0, 1}
assert abs(bernoulli_running_mean[-1] - repeated_probability) < 0.06


### Do not reverse the conditional probability

The earlier feature histogram estimates the shape of
$X\mid Y=\text{class}$: how a measurement is distributed within each observed
class. Logistic regression estimates $P(Y=1\mid X=x)$: how event probability
changes after a measurement is known. These conditional directions are not the
same.

Bayes' rule connects them through the event prevalence and the marginal feature
distribution. Consequently, a prevalence change can alter
$P(Y=1\mid X=x)$ even when the class-conditional feature distributions remain
similar. This is one reason a fitted probability model must name its applicable
population and why deployment calibration can drift.

**Checkpoint:** a measurement is common among malignant observations. Does that
fact alone imply a high malignant-event probability? Identify the two additional
distributions needed to answer.


## 3. Bernoulli likelihood produces binary cross-entropy

Conditionally on $\boldsymbol x_i$, assume

$$
Y_i\mid\boldsymbol x_i\sim\operatorname{Bernoulli}(p_i),
\qquad p_i=\sigma(z_i).
$$

The probability mass assigned to the observed $y_i\in\{0,1\}$ is

$$
P(Y_i=y_i\mid\boldsymbol x_i)=p_i^{y_i}(1-p_i)^{1-y_i}.
$$

Conditional independence gives a **product** likelihood. Taking a logarithm
turns that product into a sum. Maximizing likelihood is equivalent to minimizing
the average negative log-likelihood

$$
J(\boldsymbol w,b)
=-\frac1n\sum_{i=1}^n
\left[y_i\log p_i+(1-y_i)\log(1-p_i)\right].
$$

This is binary cross-entropy, logistic loss, or log loss. It is not chosen only
because it “looks smooth”; it is the proper likelihood objective for the stated
conditional Bernoulli model.


In [ ]:
# Avoid endpoints because log(0) is infinite, exactly as the mathematics says.
probability_grid = np.linspace(0.001, 0.999, 500)
loss_if_event = -np.log(probability_grid)
loss_if_no_event = -np.log(1.0 - probability_grid)

fig, ax = plt.subplots(figsize=(8.5, 4.8))
ax.plot(probability_grid, loss_if_event, label="observed y = 1", linewidth=2.3)
ax.plot(probability_grid, loss_if_no_event, label="observed y = 0", linewidth=2.3)
ax.set(
    xlabel="modeled probability p = P(Y = 1 | x)",
    ylabel="per-example log loss",
    title="Confidently wrong probabilities receive an unbounded penalty",
    ylim=(0, 7),
)
ax.legend()
plt.show()

assert loss_if_event[0] > loss_if_event[-1]
assert loss_if_no_event[0] < loss_if_no_event[-1]


### Likelihood asks which parameter values explain an observed sample

Probability treats $p$ as fixed and outcomes as variable. **Likelihood** holds
the observed outcomes fixed and views the same formula as a function of the
unknown parameter. For $n$ comparable Bernoulli observations with $k$ events,

$$
L(p;y_1,\ldots,y_n)
=\prod_{i=1}^n p^{y_i}(1-p)^{1-y_i}
=p^k(1-p)^{n-k}.
$$

The maximum occurs at $\widehat p=k/n$. A dataset with eight events in ten
trials supports larger values of $p$ than a dataset with two events in ten, but
neither sample identifies $p$ without uncertainty. Logistic regression replaces
the one shared parameter with $p_i=\sigma(\boldsymbol w^T\boldsymbol x_i+b)$.
The product still combines evidence across conditionally independent rows.


In [ ]:
def bernoulli_likelihood(
    probability: np.ndarray, *, events: int, trials: int
) -> np.ndarray:
    """Evaluate a Bernoulli sample likelihood over candidate probabilities."""

    if not 0 <= events <= trials:
        raise ValueError("events must lie between zero and trials")
    return probability**events * (1.0 - probability) ** (trials - events)


# Compare samples with the same size so the event count drives the shift.
candidate_probability = np.linspace(0.001, 0.999, 600)
fig, ax = plt.subplots(figsize=(8.5, 4.8))
for event_count, color in ((2, "#1f77b4"), (5, "#7b3294"), (8, "#c44e52")):
    likelihood_curve = bernoulli_likelihood(
        candidate_probability, events=event_count, trials=10
    )
    # Normalize only for shape comparison; the maximizing p is unchanged.
    likelihood_curve = likelihood_curve / likelihood_curve.max()
    ax.plot(
        candidate_probability,
        likelihood_curve,
        color=color,
        linewidth=2.2,
        label=f"k={event_count} events in n=10",
    )
    # Mark the sample proportion that maximizes each likelihood.
    ax.axvline(event_count / 10, color=color, alpha=0.35, linestyle="--")
ax.set(
    xlabel="candidate Bernoulli parameter p",
    ylabel="relative likelihood",
    title="Observed event counts shift the supported probability values",
)
ax.legend()
plt.show()

assert np.isclose(candidate_probability[np.argmax(
    bernoulli_likelihood(candidate_probability, events=5, trials=10)
)], 0.5, atol=0.002)


### Why log loss rewards honest probabilities

Suppose the true event probability for a stable population is $q$, while a
model reports $p$. Before observing the next outcome, its expected log loss is

$$
R_q(p)=-q\log p-(1-q)\log(1-p).
$$

Differentiation gives

$$
R_q'(p)=\frac{p-q}{p(1-p)},
$$

so the unique minimum on $(0,1)$ occurs at $p=q$. Equivalently,

$$
R_q(p)-R_q(q)
=D_{\mathrm{KL}}\!\left(
\operatorname{Bernoulli}(q)\,\Vert\,
\operatorname{Bernoulli}(p)
\right)\ge0.
$$

Log loss is therefore a **strictly proper scoring rule**: in expectation, a
forecaster cannot improve the score by systematically reporting a probability
different from the probability it actually believes. This does not guarantee
that the logistic model can represent the true $q(\boldsymbol x)$ or that the
training population matches deployment.


In [ ]:
def expected_log_loss(
    reported_probability: np.ndarray, true_probability: float
) -> np.ndarray:
    """Return expected Bernoulli log loss under a declared true probability."""

    # Reject an invalid Bernoulli parameter before evaluating logarithms.
    if not 0.0 < true_probability < 1.0:
        raise ValueError("true_probability must lie strictly between 0 and 1")
    reported = np.asarray(reported_probability, dtype=float)
    return -(
        true_probability * np.log(reported)
        + (1.0 - true_probability) * np.log(1.0 - reported)
    )


# Each curve must attain its minimum where the report equals the true q.
reported_probability = np.linspace(0.001, 0.999, 700)
fig, ax = plt.subplots(figsize=(8.5, 4.8))
for true_probability, color in ((0.2, "#1f77b4"), (0.5, "#7b3294"), (0.8, "#c44e52")):
    risk_curve = expected_log_loss(
        reported_probability, true_probability
    )
    ax.plot(
        reported_probability,
        risk_curve,
        color=color,
        linewidth=2.2,
        label=f"true q={true_probability:.1f}",
    )
    ax.scatter(
        [true_probability],
        [expected_log_loss(np.array([true_probability]), true_probability)[0]],
        color=color,
        s=45,
        zorder=3,
    )
ax.set(
    xlabel="reported probability p",
    ylabel="expected log loss",
    title="Expected log loss is minimized by reporting the true probability",
    ylim=(0, 2.2),
)
ax.legend()
plt.show()

# Recover each declared q numerically as the corresponding risk minimizer.
for true_probability in (0.2, 0.5, 0.8):
    minimizing_report = reported_probability[np.argmin(
        expected_log_loss(reported_probability, true_probability)
    )]
    assert np.isclose(minimizing_report, true_probability, atol=0.002)


## 4. Derive the gradient: the cancellation that matters

For one observation, let

$$
\ell_i=-y_i\log p_i-(1-y_i)\log(1-p_i),
\quad p_i=\sigma(z_i),
\quad z_i=b+\boldsymbol w^T\boldsymbol x_i.
$$

Differentiate each link:

$$
\frac{\partial \ell_i}{\partial p_i}
=\frac{p_i-y_i}{p_i(1-p_i)},
\qquad
\frac{\partial p_i}{\partial z_i}=p_i(1-p_i),
$$

so the factors cancel:

$$
\frac{\partial\ell_i}{\partial z_i}=p_i-y_i.
$$

Since $\partial z_i/\partial\boldsymbol w=\boldsymbol x_i$ and
$\partial z_i/\partial b=1$,

$$
\nabla_{\boldsymbol w}J
=\frac1nX^T(\boldsymbol p-\boldsymbol y),
\qquad
\frac{\partial J}{\partial b}
=\frac1n\mathbf 1^T(\boldsymbol p-\boldsymbol y).
$$

This simple residual form depends on the sigmoid–cross-entropy pairing. A
sigmoid trained with squared error has the extra factor $p_i(1-p_i)$ and can
learn very slowly when it is confidently wrong.


In [ ]:
# For y=1, compare derivatives with respect to the logit z.
gradient_logits = np.linspace(-12, 8, 500)
gradient_probabilities = sigmoid(gradient_logits)
# Cross-entropy cancels the sigmoid derivative; squared error does not.
cross_entropy_logit_gradient = gradient_probabilities - 1.0
squared_error_logit_gradient = (
    (gradient_probabilities - 1.0)
    * gradient_probabilities
    * (1.0 - gradient_probabilities)
)

# Render both gradients on the same scale to reveal saturation.
fig, ax = plt.subplots(figsize=(8.5, 4.8))
ax.plot(
    gradient_logits,
    cross_entropy_logit_gradient,
    label="cross-entropy: p - y",
    linewidth=2.3,
)
ax.plot(
    gradient_logits,
    squared_error_logit_gradient,
    label="squared error: (p - y)p(1-p)",
    linewidth=2.3,
)
ax.axhline(0, color="black", linewidth=0.8)
ax.set(
    xlabel="logit z when y = 1",
    ylabel="∂loss / ∂z",
    title="The likelihood gradient still reacts when the model is confidently wrong",
)
ax.legend()
plt.show()

assert cross_entropy_logit_gradient[0] < -0.99
assert abs(squared_error_logit_gradient[0]) < 1e-4


### Why does the code use `logaddexp(0, z)`?

This is not an additional modeling assumption. NumPy defines

$$
\operatorname{logaddexp}(a,b)=\log\left(e^a+e^b\right).
$$

Therefore

$$
\operatorname{logaddexp}(0,z)=\log\left(e^0+e^z\right)=\log(1+e^z).
$$

The `0` is the **first argument** of a two-input numerical operation; it is not a coefficient or a new variable. NumPy computes this expression with a stable algorithm equivalent to

$$
\max(0,z)+\log\left(1+e^{-|z|}\right),
$$

which avoids constructing the enormous number $e^z$ when $z$ is large. This is a useful software-engineering principle: an algebraically correct formula may still need an equivalent numerical form to remain correct on a computer.

**Quick check.** Substitute $a=0$ into the definition above. Where does the `1` in $\log(1+e^z)$ come from?


In [ ]:
def binary_log_loss_from_logits(
    logits: np.ndarray, targets: np.ndarray
) -> float:
    """Return mean Bernoulli negative log-likelihood from affine scores.

    Parameters
    ----------
    logits : numpy.ndarray of shape (n_samples,)
        Finite affine scores.
    targets : numpy.ndarray of shape (n_samples,)
        Binary observations encoded as zero or one.

    Returns
    -------
    float
        Mean unregularized log loss.

    Notes
    -----
    ``logaddexp(0, z) - y*z`` is algebraically equal to binary
    cross-entropy but remains finite for extreme logits.
    """

    # Normalize and validate the mathematical domain before evaluating a loss.
    score = np.asarray(logits, dtype=float)
    target = np.asarray(targets, dtype=float)
    if score.shape != target.shape:
        raise ValueError("logits and targets must have identical shapes")
    if not np.isin(target, [0.0, 1.0]).all():
        raise ValueError("targets must contain only 0 and 1")
    # Use the softplus form so extreme logits never require log(0).
    return float(np.mean(np.logaddexp(0.0, score) - target * score))


def logistic_loss_and_gradient(
    design: np.ndarray, targets: np.ndarray, parameters: np.ndarray
) -> tuple[float, np.ndarray]:
    """Evaluate unregularized logistic loss and its batch gradient.

    Parameters
    ----------
    design : numpy.ndarray of shape (n_samples, n_parameters)
        Design matrix whose first column may represent the intercept.
    targets : numpy.ndarray of shape (n_samples,)
        Binary event indicators.
    parameters : numpy.ndarray of shape (n_parameters,)
        Bias and feature weights aligned with ``design``.

    Returns
    -------
    loss : float
        Mean binary cross-entropy.
    gradient : numpy.ndarray of shape (n_parameters,)
        Gradient in the same coordinate order as ``parameters``.
    """

    # Convert interfaces once, then validate row and parameter alignment.
    matrix = np.asarray(design, dtype=float)
    target = np.asarray(targets, dtype=float)
    weights = np.asarray(parameters, dtype=float)
    if matrix.ndim != 2 or matrix.shape[0] != target.size:
        raise ValueError("design rows must align with targets")
    if matrix.shape[1] != weights.size:
        raise ValueError("design columns must align with parameters")

    # Forward pass: affine score, sigmoid probability, and stable likelihood loss.
    logits = matrix @ weights
    probabilities = sigmoid(logits)
    loss = binary_log_loss_from_logits(logits, target)
    # Backward pass: average each input times its probability residual.
    gradient = matrix.T @ (probabilities - target) / len(target)
    return loss, gradient


### Verify the derivative before trusting the optimizer

A derivation can contain an algebra mistake and code can contain an indexing
mistake. A centered finite difference independently approximates

$$
\frac{\partial J}{\partial\theta_j}
\approx\frac{J(\boldsymbol\theta+h\boldsymbol e_j)
-J(\boldsymbol\theta-h\boldsymbol e_j)}{2h}.
$$

Agreement at a moderate $h$ is strong local evidence for the implementation.
It is not proof for every input, so reusable code also needs unit tests for
shapes, target values, extreme logits, and known examples.


In [ ]:
def finite_difference_gradient(
    design: np.ndarray,
    targets: np.ndarray,
    parameters: np.ndarray,
    step: float = 1e-6,
) -> np.ndarray:
    """Approximate the logistic-loss gradient by centered differences."""

    # Perturb one coordinate at a time while holding all others fixed.
    approximation = np.empty_like(parameters, dtype=float)
    for index in range(len(parameters)):
        perturbation = np.zeros_like(parameters, dtype=float)
        perturbation[index] = step
        loss_plus, _ = logistic_loss_and_gradient(
            design, targets, parameters + perturbation
        )
        loss_minus, _ = logistic_loss_and_gradient(
            design, targets, parameters - perturbation
        )
        approximation[index] = (loss_plus - loss_minus) / (2.0 * step)
    return approximation


# Use a small, non-symmetric example so sign and intercept errors are visible.
check_design = np.array(
    [[1.0, -1.2, 0.4], [1.0, 0.3, -0.8], [1.0, 1.7, 1.1], [1.0, 0.2, 0.9]]
)
check_targets = np.array([0.0, 1.0, 1.0, 0.0])
check_parameters = np.array([0.15, -0.35, 0.22])
_, analytic_gradient = logistic_loss_and_gradient(
    check_design, check_targets, check_parameters
)
numerical_gradient = finite_difference_gradient(
    check_design, check_targets, check_parameters
)
# Reduce coordinatewise disagreement to one auditable scalar.
gradient_check_error = np.max(np.abs(analytic_gradient - numerical_gradient))

assert gradient_check_error < 1e-8
print(f"maximum gradient-check error: {gradient_check_error:.3e}")


## 5. Geometry: convex loss, but not always a finite optimum

Let $\widetilde X$ include an intercept column and let
$W=\operatorname{diag}(p_i(1-p_i))$. The Hessian is

$$
\nabla^2J(\boldsymbol\theta)
=\frac1n\widetilde X^TW\widetilde X.
$$

For any vector $\boldsymbol v$,

$$
\boldsymbol v^T\nabla^2J\boldsymbol v
=\frac1n\bigl(\widetilde X\boldsymbol v\bigr)^T
W\bigl(\widetilde X\boldsymbol v\bigr)\ge 0,
$$

because $p_i(1-p_i)\ge0$. Therefore unregularized logistic loss is convex.
Every finite local minimizer is global. Strict convexity and uniqueness require
additional rank and overlap conditions.

If the classes are perfectly linearly separable, the unregularized likelihood
may improve as $\lVert\boldsymbol w\rVert\to\infty$ without attaining a finite
maximum-likelihood estimate. Regularization is therefore statistical control
and an existence/stability device, not merely an optimizer trick.


In [ ]:
# Fit the scaler on training data only; validation and test use its stored state.
one_feature_scaler = StandardScaler().fit(train_data[[one_feature]])
train_feature_scaled = one_feature_scaler.transform(train_data[[one_feature]])
validation_feature_scaled = one_feature_scaler.transform(
    validation_data[[one_feature]]
)

# An intercept column makes theta = [b, w] one vector for calculus and code.
train_design = np.column_stack(
    [np.ones(len(train_feature_scaled)), train_feature_scaled[:, 0]]
)
validation_design = np.column_stack(
    [np.ones(len(validation_feature_scaled)), validation_feature_scaled[:, 0]]
)
train_target = train_data["malignant_event"].to_numpy(dtype=float)
validation_target = validation_data["malignant_event"].to_numpy(dtype=float)

initial_parameters = np.zeros(train_design.shape[1])
initial_loss, initial_gradient = logistic_loss_and_gradient(
    train_design, train_target, initial_parameters
)
initial_probabilities = sigmoid(train_design @ initial_parameters)
# Build X^T W X / n explicitly and inspect its eigenvalues.
initial_weights = initial_probabilities * (1.0 - initial_probabilities)
initial_hessian = (
    train_design.T @ (initial_weights[:, None] * train_design) / len(train_target)
)
initial_eigenvalues = np.linalg.eigvalsh(initial_hessian)

assert np.all(initial_eigenvalues >= -1e-12)
print(f"initial loss: {initial_loss:.4f}")
print("initial gradient:", initial_gradient)
print("Hessian eigenvalues:", initial_eigenvalues)


## 6. Train the neuron with batch gradient descent

With $\boldsymbol\theta=[b,w_1,\ldots,w_d]^T$, one batch update is

$$
\boldsymbol\theta_{t+1}
=\boldsymbol\theta_t-\eta\,
\frac1n\widetilde X^T
\left(\sigma(\widetilde X\boldsymbol\theta_t)-\boldsymbol y\right).
$$

The model is the affine-score-plus-sigmoid map. Gradient descent is only the
optimizer used to estimate its parameters. The optimizer should return
termination evidence explaining why it stopped rather than an unexplained last
iterate.


In [ ]:
@dataclass(frozen=True)
class LogisticFitResult:
    """Immutable evidence returned by batch logistic-neuron training."""

    parameters: np.ndarray
    parameter_history: np.ndarray
    loss_history: np.ndarray
    gradient_norm_history: np.ndarray
    converged: bool
    iterations: int


def fit_logistic_neuron(
    design: np.ndarray,
    targets: np.ndarray,
    *,
    learning_rate: float = 0.2,
    max_iterations: int = 10_000,
    tolerance: float = 1e-9,
) -> LogisticFitResult:
    """Fit an unregularized logistic neuron by batch gradient descent.

    Parameters
    ----------
    design : numpy.ndarray of shape (n_samples, n_parameters)
        Numeric design matrix, including an intercept column when desired.
    targets : numpy.ndarray of shape (n_samples,)
        Binary event indicators.
    learning_rate : float, default=0.2
        Positive fixed step size.
    max_iterations : int, default=10000
        Positive safety limit on parameter updates.
    tolerance : float, default=1e-9
        Convergence threshold for the Euclidean gradient norm.

    Returns
    -------
    LogisticFitResult
        Parameters, optimization traces, and termination evidence.

    Raises
    ------
    ValueError
        If an optimization control is outside its documented domain.
    """

    # Reject controls that make the optimizer contract meaningless.
    if learning_rate <= 0:
        raise ValueError("learning_rate must be positive")
    if max_iterations <= 0:
        raise ValueError("max_iterations must be positive")
    if tolerance <= 0:
        raise ValueError("tolerance must be positive")

    # Store full traces because this is an instructional, inspectable optimizer.
    parameters = np.zeros(design.shape[1], dtype=float)
    parameter_history: list[np.ndarray] = []
    loss_history: list[float] = []
    gradient_norm_history: list[float] = []

    # Each iteration uses all rows, so one update equals one batch gradient.
    for _iteration in range(1, max_iterations + 1):
        loss, gradient = logistic_loss_and_gradient(design, targets, parameters)
        gradient_norm = float(np.linalg.norm(gradient))

        # Record the state before any update so traces align exactly.
        parameter_history.append(parameters.copy())
        loss_history.append(loss)
        gradient_norm_history.append(gradient_norm)

        if gradient_norm <= tolerance:
            converged = True
            break
        parameters = parameters - learning_rate * gradient
    else:
        converged = False

    # Package the final state and its evidence in an immutable result object.
    return LogisticFitResult(
        parameters=parameters,
        parameter_history=np.asarray(parameter_history),
        loss_history=np.asarray(loss_history),
        gradient_norm_history=np.asarray(gradient_norm_history),
        converged=converged,
        iterations=_iteration,
    )


In [ ]:
# Fit the transparent one-feature neuron used in the derivation.
one_feature_fit = fit_logistic_neuron(
    train_design,
    train_target,
    learning_rate=0.2,
    max_iterations=12_000,
    tolerance=1e-9,
)

# Compute independent evidence on the validation partition.
validation_logits = validation_design @ one_feature_fit.parameters
validation_probabilities = sigmoid(validation_logits)
validation_log_loss = binary_log_loss_from_logits(
    validation_logits, validation_target
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(one_feature_fit.loss_history, color="#2a9d8f")
axes[0].set(xlabel="iteration", ylabel="training log loss", title="Objective trace")
axes[1].semilogy(one_feature_fit.gradient_norm_history, color="#e76f51")
axes[1].set(
    xlabel="iteration", ylabel="gradient norm", title="First-order convergence evidence"
)
plt.tight_layout()
plt.show()

# Treat both termination and monotone descent as executable expectations.
assert one_feature_fit.converged
assert one_feature_fit.loss_history[-1] < one_feature_fit.loss_history[0]
assert np.all(np.diff(one_feature_fit.loss_history) <= 1e-12)
print(
    f"iterations={one_feature_fit.iterations:,}; "
    f"validation log loss={validation_log_loss:.4f}"
)


In [ ]:
# Build a raw-feature domain, then transform it with training-only statistics.
raw_curve_domain = np.linspace(
    train_data[one_feature].min(), train_data[one_feature].max(), 400
).reshape(-1, 1)
scaled_curve_domain = one_feature_scaler.transform(raw_curve_domain)
curve_design = np.column_stack(
    [np.ones(len(scaled_curve_domain)), scaled_curve_domain[:, 0]]
)
curve_probability = sigmoid(curve_design @ one_feature_fit.parameters)

# Plot labels, fitted probability, and the default 0.5 threshold separately.
rng = np.random.default_rng(RANDOM_SEED)
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(
    train_data[one_feature],
    train_target + rng.normal(0.0, 0.018, len(train_target)),
    alpha=0.32,
    s=22,
    label="training labels (jittered)",
)
ax.plot(raw_curve_domain[:, 0], curve_probability, color="#6a3d9a", linewidth=2.8)
ax.axhline(0.5, color="black", linestyle="--", linewidth=1.0, label="0.5 policy")
ax.set(
    xlabel="worst concave points",
    ylabel="P(malignant event | measurement)",
    title="One logistic neuron learns a smooth conditional-probability curve",
)
ax.legend()
plt.show()

# The model range is open for finite logits even though floating point can
# round sufficiently extreme values.
assert np.all((curve_probability > 0) & (curve_probability < 1))


### From individual probabilities to frequency evidence

One observed outcome cannot reveal whether its probability was correct. A
well-calibrated model may assign $p=0.8$ to an observation whose outcome is
zero; that outcome should occur about $20\%$ of the time among genuinely
comparable $p=0.8$ cases.

To obtain evidence, group untouched observations with similar predicted
probabilities and compare each group's mean prediction with its event fraction.
The event count in a bin is binomial only under additional independence and
homogeneity assumptions. A Wilson interval visualizes sampling uncertainty
without pretending that a small bin supplies an exact event rate.

The second panel below asks a different question: do observed classes tend to
receive different probability distributions? That is discrimination, not calibration.
A useful probability model needs both ideas kept distinct.


In [ ]:
# Build a validation-only table so frequency evidence is independent of fitting.
validation_probability_frame = pd.DataFrame(
    {
        "probability": validation_probabilities,
        "event": validation_target.astype(int),
    },
    index=validation_data.index,
)
validation_probability_frame["probability_bin"] = pd.qcut(
    validation_probability_frame["probability"], q=6, duplicates="drop"
)
validation_probability_summary = (
    validation_probability_frame.groupby("probability_bin", observed=True)
    .agg(
        rows=("event", "size"),
        mean_probability=("probability", "mean"),
        event_rate=("event", "mean"),
    )
    .reset_index(drop=True)
)

# Compute a 95% Wilson interval for each observed binomial proportion.
z_score = 1.96
row_count = validation_probability_summary["rows"].to_numpy(dtype=float)
event_rate = validation_probability_summary["event_rate"].to_numpy()
wilson_denominator = 1.0 + z_score**2 / row_count
wilson_center = (
    event_rate + z_score**2 / (2.0 * row_count)
) / wilson_denominator
wilson_half_width = (
    z_score
    / wilson_denominator
    * np.sqrt(
        event_rate * (1.0 - event_rate) / row_count
        + z_score**2 / (4.0 * row_count**2)
    )
)

# Plot frequency agreement and class-conditional probability distributions.
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))
axes[0].errorbar(
    validation_probability_summary["mean_probability"],
    wilson_center,
    yerr=wilson_half_width,
    fmt="o",
    capsize=4,
    color="#2a9d8f",
    label="observed fraction with 95% Wilson interval",
)
axes[0].plot([0, 1], [0, 1], color="black", linestyle="--", label="ideal agreement")
axes[0].set(
    xlim=(0, 1),
    ylim=(0, 1),
    xlabel="mean modeled probability in bin",
    ylabel="observed event fraction",
    title="Validation frequency evidence is uncertain",
)
axes[0].legend(fontsize=8)

probability_bins = np.linspace(0.0, 1.0, 13)
for event_value, label, color in (
    (0, "observed benign", "#1f77b4"),
    (1, "observed malignant", "#c44e52"),
):
    axes[1].hist(
        validation_probability_frame.loc[
            validation_probability_frame["event"] == event_value, "probability"
        ],
        bins=probability_bins,
        alpha=0.55,
        label=label,
        color=color,
    )
axes[1].set(
    xlim=(0, 1),
    xlabel="modeled malignant-event probability",
    ylabel="validation rows",
    title="Discrimination: probability distributions overlap",
)
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

assert validation_probability_summary["rows"].sum() == len(validation_data)
assert np.all(wilson_half_width > 0)


In [ ]:
# Evaluate a local loss surface around the optimum for two parameters [b, w].
fitted_bias, fitted_weight = one_feature_fit.parameters
bias_grid = np.linspace(fitted_bias - 3.5, fitted_bias + 3.5, 100)
weight_grid = np.linspace(max(0.05, fitted_weight - 3.5), fitted_weight + 3.5, 100)
loss_surface = np.empty((len(weight_grid), len(bias_grid)))
# Recompute the true objective at each coordinate rather than interpolating.
for row, weight in enumerate(weight_grid):
    for column, bias in enumerate(bias_grid):
        loss_surface[row, column], _ = logistic_loss_and_gradient(
            train_design, train_target, np.array([bias, weight])
        )

# Thin the optimizer trace so the path is visible rather than painted solid.
trace_step = max(1, len(one_feature_fit.parameter_history) // 45)
visible_trace = one_feature_fit.parameter_history[::trace_step]
fig, ax = plt.subplots(figsize=(8, 5.5))
contours = ax.contour(bias_grid, weight_grid, loss_surface, levels=22, cmap="viridis")
ax.plot(
    visible_trace[:, 0],
    visible_trace[:, 1],
    marker="o",
    markersize=2.5,
    color="#d62728",
    linewidth=1.2,
    label="gradient-descent path",
)
ax.scatter([fitted_bias], [fitted_weight], marker="*", s=150, color="black")
ax.clabel(contours, inline=True, fontsize=7)
ax.set(xlabel="bias b", ylabel="scaled-feature weight w", title="Convex loss geometry")
ax.legend()
plt.show()

assert loss_surface.min() <= one_feature_fit.loss_history[-1] + 0.01


## 7. In what sense is this “like linear regression”?

Both one-neuron models begin with the same affine score
$\widetilde X\boldsymbol\theta$. Their gradients share an outer algebraic
form:

| model | output | average objective | parameter gradient |
|---|---|---|---|
| linear neuron | $\widehat{\boldsymbol y}=\widetilde X\boldsymbol\theta$ | $\lVert\widehat{\boldsymbol y}-\boldsymbol y\rVert_2^2/(2n)$ | $\widetilde X^T(\widehat{\boldsymbol y}-\boldsymbol y)/n$ |
| logistic neuron | $\boldsymbol p=\sigma(\widetilde X\boldsymbol\theta)$ | Bernoulli negative log-likelihood | $\widetilde X^T(\boldsymbol p-\boldsymbol y)/n$ |

The resemblance is real: “input times output-layer error” is the single-layer
form of back-propagation. The models are not interchangeable:

- identity and sigmoid activations define different response functions;
- Gaussian-style squared loss and Bernoulli log loss encode different data models;
- linear regression has Hessian $\widetilde X^T\widetilde X/n$;
- logistic regression has parameter-dependent Hessian $\widetilde X^TW\widetilde X/n$;
- ordinary least squares has a closed-form projection solution, while logistic
  maximum likelihood is normally solved iteratively.

So logistic regression is not linear regression followed by clipping. It is a
different likelihood model built on the same affine computational skeleton.


In [ ]:
# C=np.inf asks scikit-learn for the same unregularized one-feature objective.
reference_neuron = LogisticRegression(
    C=np.inf,
    solver="lbfgs",
    max_iter=10_000,
    tol=1e-12,
).fit(train_feature_scaled, train_target)
reference_parameters = np.concatenate(
    [reference_neuron.intercept_, reference_neuron.coef_.ravel()]
)
reference_validation_probability = reference_neuron.predict_proba(
    validation_feature_scaled
)[:, 1]

# Agreement checks the optimizer and parameter ordering for this exact contract.
parameter_error = np.max(
    np.abs(one_feature_fit.parameters - reference_parameters)
)
probability_error = np.max(
    np.abs(validation_probabilities - reference_validation_probability)
)

# Tight agreement is expected because scaling, intercept, and regularization match.
assert parameter_error < 1e-5
assert probability_error < 1e-6
print(f"maximum parameter difference: {parameter_error:.3e}")
print(f"maximum validation-probability difference: {probability_error:.3e}")


## 8. Multiple inputs and a professional fitted pipeline

A neuron may have many inputs while retaining one output:

$$
z=b+w_1x_1+\cdots+w_dx_d,
\qquad p=\sigma(z).
$$

We now use six predeclared measurements. A `Pipeline` binds a training-fitted
`StandardScaler` to a regularized `LogisticRegression`. Scaling changes the
coordinate system and optimizer conditioning, not the set of unregularized
linear decision boundaries. L2 regularization changes the objective by
penalizing large feature weights and is enabled explicitly through finite
`C=1.0`.

This model is a stronger classroom baseline, not a clinical release candidate.
The dataset lacks site, time, demographic, and acquisition-process fields
needed for consequential validation.


In [ ]:
feature_names = [
    "mean_radius",
    "mean_texture",
    "mean_smoothness",
    "mean_concave_points",
    "worst_radius",
    "worst_concave_points",
]

# The baseline uses only event prevalence learned from training targets.
probability_baseline = DummyClassifier(strategy="prior").fit(
    train_data[feature_names], train_data["malignant_event"]
)
baseline_validation_probability = probability_baseline.predict_proba(
    validation_data[feature_names]
)[:, list(probability_baseline.classes_).index(1)]

# The pipeline prevents prediction with unscaled values and scaler leakage.
logistic_pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=1.0, solver="lbfgs", max_iter=2_000, tol=1e-10),
)
logistic_pipeline.fit(
    train_data[feature_names], train_data["malignant_event"]
)
model_validation_probability = logistic_pipeline.predict_proba(
    validation_data[feature_names]
)[:, 1]

# Verify that scaler state came only from the training feature matrix.
fitted_scaler = logistic_pipeline.named_steps["standardscaler"]
assert np.allclose(
    fitted_scaler.mean_, train_data[feature_names].mean().to_numpy()
)
assert not np.allclose(
    fitted_scaler.mean_, observations[feature_names].mean().to_numpy()
)


In [ ]:
def evaluate_binary_probabilities(
    targets: pd.Series | np.ndarray,
    probabilities: np.ndarray,
    *,
    threshold: float = 0.5,
) -> pd.Series:
    """Summarize probability, ranking, and threshold-dependent evidence.

    Parameters
    ----------
    targets : array-like of shape (n_samples,)
        Binary event indicators, where one names the declared event.
    probabilities : numpy.ndarray of shape (n_samples,)
        Modeled probabilities for the event encoded as one.
    threshold : float, default=0.5
        Decision cutoff used only for threshold-dependent measures.

    Returns
    -------
    pandas.Series
        Sample context, proper scores, ranking evidence, confusion counts,
        and rates with their conditioning denominators made explicit.

    Raises
    ------
    ValueError
        If inputs have incompatible shapes or invalid values.
    """

    # Normalize and validate the public inputs before computing evidence.
    actual = np.asarray(targets, dtype=int)
    probability = np.asarray(probabilities, dtype=float)
    if (
        actual.ndim != 1
        or probability.ndim != 1
        or actual.shape != probability.shape
    ):
        raise ValueError(
            "targets and probabilities must be one-dimensional and aligned"
        )
    if not np.all(np.isin(actual, [0, 1])):
        raise ValueError("targets must contain only zero and one")
    if not 0.0 < threshold < 1.0:
        raise ValueError("threshold must lie strictly between 0 and 1")
    if not np.all(np.isfinite(probability)):
        raise ValueError("probabilities must be finite")
    if not np.all((probability >= 0.0) & (probability <= 1.0)):
        raise ValueError("probabilities must lie in [0, 1]")

    # Apply the action policy only after probability metrics are defined.
    decision = (probability >= threshold).astype(int)
    true_negative, false_positive, false_negative, true_positive = (
        confusion_matrix(actual, decision, labels=[0, 1]).ravel()
    )

    def safe_rate(numerator: int, denominator: int) -> float:
        """Return a rate or NaN when its conditioning set is empty."""

        return float(numerator / denominator) if denominator else np.nan

    actual_positive = true_positive + false_negative
    actual_negative = true_negative + false_positive
    predicted_positive = true_positive + false_positive
    predicted_negative = true_negative + false_negative
    sensitivity = safe_rate(true_positive, actual_positive)
    specificity = safe_rate(true_negative, actual_negative)

    # Return named evidence so downstream reports cannot depend on row order.
    return pd.Series(
        {
            "rows": len(actual),
            "event_prevalence": actual.mean(),
            "log_loss": log_loss(actual, probability, labels=[0, 1]),
            "brier_score": brier_score_loss(actual, probability),
            "roc_auc": roc_auc_score(actual, probability),
            "threshold": threshold,
            "predicted_positive_rate": decision.mean(),
            "accuracy": accuracy_score(actual, decision),
            "sensitivity_recall": sensitivity,
            "specificity": specificity,
            "balanced_accuracy": np.nanmean([sensitivity, specificity]),
            "precision_ppv": safe_rate(true_positive, predicted_positive),
            "negative_predictive_value": safe_rate(true_negative, predicted_negative),
            "false_positive_rate": safe_rate(false_positive, actual_negative),
            "false_negative_rate": safe_rate(false_negative, actual_positive),
            "true_negative": true_negative,
            "false_positive": false_positive,
            "false_negative": false_negative,
            "true_positive": true_positive,
        }
    )


validation_evidence = pd.DataFrame(
    {
        "prevalence baseline": evaluate_binary_probabilities(
            validation_data["malignant_event"], baseline_validation_probability
        ),
        "regularized logistic pipeline": evaluate_binary_probabilities(
            validation_data["malignant_event"], model_validation_probability
        ),
    }
)
assert (
    validation_evidence.loc["log_loss", "regularized logistic pipeline"]
    < validation_evidence.loc["log_loss", "prevalence baseline"]
)
validation_evidence

## 9. Measuring model performance professionally

“How accurate is the model?” is not yet a well-posed question. A professional
evaluation names the **claim**, **population**, **sample**, **baseline**, and
**decision policy** before choosing a number.

Classification evidence has at least three layers:

| Layer | Question | Examples | Needs a threshold? |
|---|---|---|---|
| probability quality | Did the model assign useful probabilities to what occurred? | log loss, Brier score, reliability evidence | no |
| discrimination | Do event cases tend to rank above non-event cases? | ROC AUC, precision-recall curve | no, for the full curve |
| decision performance | What happened under one action rule? | confusion counts, sensitivity, specificity, precision, cost, capacity | yes |

No layer replaces the others. A model can rank well but be badly calibrated;
it can estimate probabilities well yet perform poorly under an unsuitable
threshold; and it can attain high accuracy by predicting only the common class.

Always begin with a meaningful baseline. Here the prevalence model asks whether
the fitted pipeline improves on simply reporting the training event frequency.
In deployment, the relevant baseline may be the current laboratory workflow,
an existing scoring rule, or “send every case for review.”

**Checkpoint:** Which metric would you use to support each claim: accurate
probabilities, useful ranking, and an acceptable number of missed malignant
events? Why can no single metric support all three claims?

### A confusion matrix contains four counts, not one score

Once a threshold $t$ defines the action $\widehat Y=\mathbf 1\{p\ge t\}$,
every validation row enters exactly one cell:

| | predicted non-event $\widehat Y=0$ | predicted event $\widehat Y=1$ |
|---|---:|---:|
| actual non-event $Y=0$ | true negative (TN) | false positive (FP) |
| actual event $Y=1$ | false negative (FN) | true positive (TP) |

The word “positive” refers to the declared event—not to a good outcome. The
word “false” means the action disagrees with the recorded label. It does not
prove that the label, measurement, or sampling process is correct.

Rates answer different conditional questions:

$$
\text{sensitivity}=\frac{TP}{TP+FN}=P(\widehat Y=1\mid Y=1),
\qquad
\text{specificity}=\frac{TN}{TN+FP}=P(\widehat Y=0\mid Y=0),
$$

$$
\text{precision}=\frac{TP}{TP+FP}=P(Y=1\mid\widehat Y=1),
\qquad
\text{NPV}=\frac{TN}{TN+FN}=P(Y=0\mid\widehat Y=0).
$$

Sensitivity conditions on actual event rows; precision conditions on predicted
event actions. Confusing their denominators reverses the conditional probability.
Report the counts with the rates so readers can see the sample size.

In [ ]:
# Fix one illustrative policy; later we will sweep rather than privilege 0.5.
illustrative_threshold = 0.5
validation_actual = validation_data["malignant_event"].to_numpy(dtype=int)
validation_decision = (
    model_validation_probability >= illustrative_threshold
).astype(int)
validation_confusion = confusion_matrix(
    validation_actual, validation_decision, labels=[0, 1]
)

# Row normalization conditions on the actual class: each row must sum to one.
actual_conditioned = validation_confusion / validation_confusion.sum(
    axis=1, keepdims=True
)
# Pair the raw evidence with its actual-class conditional interpretation.
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2))
for ax, matrix, title, number_format in (
    (axes[0], validation_confusion, "Counts: preserve denominators", "d"),
    (axes[1], actual_conditioned, "Rows normalized by actual class", ".1%"),
):
    image = ax.imshow(matrix, cmap="Blues", vmin=0)
    for row in range(2):
        for column in range(2):
            value = matrix[row, column]
            label = (
                format(int(value), number_format)
                if number_format == "d"
                else format(value, number_format)
            )
            ax.text(column, row, label, ha="center", va="center", fontsize=12)
    ax.set(
        xticks=[0, 1],
        yticks=[0, 1],
        xticklabels=["predicted benign", "predicted malignant"],
        yticklabels=["actual benign", "actual malignant"],
        xlabel="model action at threshold 0.5",
        ylabel="recorded outcome",
        title=title,
    )
    ax.tick_params(axis="x", rotation=15)
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

assert validation_confusion.sum() == len(validation_data)
assert np.allclose(actual_conditioned.sum(axis=1), 1.0)

### Performance depends on which data are being considered

A confusion matrix is a property of a **model-policy-population-sample** tuple,
not of the model alone. Consider event prevalence
$\pi=P(Y=1)$, sensitivity $s$, and specificity $c$. Then

$$
\text{precision}
=\frac{\pi s}{\pi s+(1-\pi)(1-c)},
\qquad
\text{accuracy}=\pi s+(1-\pi)c.
$$

Even if sensitivity and specificity transported unchanged, precision and
accuracy would change with prevalence. Real transport is harder: performance
can also change with disease severity, sensor calibration, laboratory site,
acquisition protocol, missingness, time, demographic composition, and the
definition or quality of the label. This broader change in the mix of cases is
sometimes called **spectrum shift**.

The repository dataset is not a random sample of every future clinical case.
Its event prevalence must not be substituted for the prevalence of a hospital,
screening program, or community. We can learn the workflow from these rows; we
cannot infer a deployment claim that the data do not support.

In [ ]:
# Hold class-conditional operating characteristics fixed as a thought experiment.
tn, fp, fn, tp = validation_confusion.ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
prevalence_grid = np.linspace(0.01, 0.99, 300)

# Predictive values and accuracy still change when the case mix changes.
precision_by_prevalence = (
    prevalence_grid * sensitivity
    / (
        prevalence_grid * sensitivity
        + (1.0 - prevalence_grid) * (1.0 - specificity)
    )
)
npv_by_prevalence = (
    (1.0 - prevalence_grid) * specificity
    / (
        (1.0 - prevalence_grid) * specificity
        + prevalence_grid * (1.0 - sensitivity)
    )
)
accuracy_by_prevalence = (
    prevalence_grid * sensitivity
    + (1.0 - prevalence_grid) * specificity
)

# Mark this validation sample without treating it as deployment prevalence.
validation_prevalence = validation_actual.mean()
fig, ax = plt.subplots(figsize=(9.5, 4.8))
ax.plot(
    prevalence_grid,
    precision_by_prevalence,
    linewidth=2.3,
    label="precision (PPV)",
)
ax.plot(
    prevalence_grid,
    npv_by_prevalence,
    linewidth=2.3,
    label="negative predictive value",
)
ax.plot(
    prevalence_grid,
    accuracy_by_prevalence,
    linewidth=2.3,
    label="accuracy",
)
ax.axvline(
    validation_prevalence,
    color="black",
    linestyle="--",
    label=f"validation prevalence = {validation_prevalence:.2f}",
)
ax.set(
    xlim=(0, 1),
    ylim=(0, 1.02),
    xlabel="event prevalence in the evaluated population",
    ylabel="metric value under fixed sensitivity/specificity",
    title="Some reported performance changes when the population changes",
)
ax.legend(ncols=2, fontsize=8)
plt.show()

assert not np.isclose(precision_by_prevalence[10], precision_by_prevalence[-10])

### A threshold sweep connects probabilities to an operational choice

Changing the threshold does not refit logistic regression. It changes the
decision policy and therefore moves observations between confusion-matrix
cells. Lower thresholds usually increase sensitivity and action volume while
reducing specificity. Precision may move in either direction because it also
depends on the evaluated case mix.

A professional threshold study should:

1. use validation data, never the sealed test set;
2. state error costs, review capacity, or another operational constraint;
3. examine the entire useful threshold range rather than only $0.5$;
4. report counts and uncertainty at candidate operating points; and
5. freeze and version the chosen policy before final evaluation.

The following cost is explicitly hypothetical: one false negative is assigned
five times the cost of one false positive. It is a teaching device, not a
clinical valuation.

In [ ]:
# Evaluate each policy against the same validation probabilities and outcomes.
threshold_grid = np.linspace(0.02, 0.98, 97)
threshold_records: list[dict[str, float]] = []
for threshold in threshold_grid:
    evidence = evaluate_binary_probabilities(
        validation_actual,
        model_validation_probability,
        threshold=float(threshold),
    )
    false_negative = evidence["false_negative"]
    false_positive = evidence["false_positive"]
    threshold_records.append(
        {
            "threshold": threshold,
            "sensitivity": evidence["sensitivity_recall"],
            "specificity": evidence["specificity"],
            "precision": evidence["precision_ppv"],
            "action_rate": evidence["predicted_positive_rate"],
            "cost_per_row": (5.0 * false_negative + false_positive)
            / evidence["rows"],
        }
    )
threshold_table = pd.DataFrame(threshold_records)

# Select on validation only, using the declared cost rather than test feedback.
selected_row = threshold_table.loc[threshold_table["cost_per_row"].idxmin()]
selected_threshold = float(selected_row["threshold"])
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.5))
for metric in ("sensitivity", "specificity", "precision", "action_rate"):
    axes[0].plot(
        threshold_table["threshold"],
        threshold_table[metric],
        linewidth=2,
        label=metric.replace("_", " "),
    )
axes[0].axvline(selected_threshold, color="black", linestyle="--")
axes[0].set(
    xlim=(0, 1),
    ylim=(0, 1.02),
    xlabel="decision threshold",
    ylabel="validation rate",
    title="Every threshold defines a different operating point",
)
axes[0].legend(fontsize=8)

# Cost combines counts only after the relative consequence is declared.
axes[1].plot(
    threshold_table["threshold"],
    threshold_table["cost_per_row"],
    linewidth=2.4,
    color="#8c2d04",
)
axes[1].scatter(
    [selected_threshold],
    [selected_row["cost_per_row"]],
    color="black",
    zorder=3,
    label=f"validation choice = {selected_threshold:.2f}",
)
axes[1].set(
    xlim=(0, 1),
    xlabel="decision threshold",
    ylabel="hypothetical cost per validation row",
    title="A decision requires a declared consequence model",
)
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

assert 0.0 < selected_threshold < 1.0

### Point estimates are not a professional performance report

Every validation metric is estimated from a finite sample. A rate with a
denominator of twelve carries less evidence than the same rate with a
denominator of twelve thousand. For a fixed, predeclared model and threshold,
binomial intervals can summarize uncertainty in sensitivity or specificity
when their assumptions are plausible. If observations are grouped by patient,
laboratory, or time, resampling must preserve those groups.

A serious report records:

- event definition, horizon, prediction unit, and intended-use population;
- dataset source, dates, sites, inclusion rules, sample sizes, and prevalence;
- split unit and all model-selection decisions;
- baseline, log loss/Brier score, ranking evidence, and calibration evidence;
- versioned threshold, confusion counts, conditional rates, costs, and capacity;
- uncertainty intervals and performance by relevant site, time, and subgroup;
- known missing slices, label limitations, and reasons deployment data may differ; and
- model, feature, software, and policy versions needed to reproduce the report.

This dataset does not include the demographic, hospital, acquisition-time, or
deployment-workflow fields needed for a credible subgroup or external-validity
audit. The professional conclusion is not to ignore those analyses; it is to
name the evidence gap.

In [ ]:
# Use a score interval that behaves better than a Wald interval near 0 and 1.
def wilson_interval(
    successes: int, trials: int, z_score: float = 1.96
) -> tuple[float, float]:
    """Return a Wilson score interval for one binomial proportion."""

    if trials <= 0:
        raise ValueError("trials must be positive")
    proportion = successes / trials
    denominator = 1.0 + z_score**2 / trials
    center = (proportion + z_score**2 / (2.0 * trials)) / denominator
    half_width = (
        z_score
        / denominator
        * np.sqrt(
            proportion * (1.0 - proportion) / trials
            + z_score**2 / (4.0 * trials**2)
        )
    )
    return center - half_width, center + half_width


# Freeze the validation-selected policy before attaching interval estimates.
selected_evidence = evaluate_binary_probabilities(
    validation_actual,
    model_validation_probability,
    threshold=selected_threshold,
)
selected_tp = int(selected_evidence["true_positive"])
selected_tn = int(selected_evidence["true_negative"])
selected_fn = int(selected_evidence["false_negative"])
selected_fp = int(selected_evidence["false_positive"])
# Preserve each conditioning denominator while organizing the report.
rate_intervals = pd.DataFrame(
    {
        "estimate": {
            "sensitivity": selected_evidence["sensitivity_recall"],
            "specificity": selected_evidence["specificity"],
            "precision": selected_evidence["precision_ppv"],
        },
        "95% Wilson interval": {
            "sensitivity": wilson_interval(
                selected_tp,
                selected_tp + selected_fn,
            ),
            "specificity": wilson_interval(
                selected_tn,
                selected_tn + selected_fp,
            ),
            "precision": wilson_interval(
                selected_tp,
                selected_tp + selected_fp,
            ),
        },
    }
)
rate_intervals

## 10. Calibration is separate evidence

The performance study above separates probability quality, ranking, and one
thresholded action rule. Calibration asks a remaining question: among comparable
cases assigned probability near $q$, does the event occur at a frequency near
$q$? **Sigmoid range does not guarantee calibration.**

In a real workflow, threshold selection requires error costs, review capacity,
abstention behavior, prevalence, and the deployment population. Thresholds and
calibration will therefore receive their own notebook. Here we plot a
reliability diagram only to demonstrate that sigmoid range does not guarantee
calibration.

In [ ]:
# Each point compares mean predicted probability with observed event frequency.
fig, ax = plt.subplots(figsize=(6.5, 5.5))
CalibrationDisplay.from_predictions(
    validation_data["malignant_event"],
    model_validation_probability,
    n_bins=8,
    strategy="quantile",
    name="logistic pipeline",
    ax=ax,
)
ax.set_title("Validation reliability diagram (small-sample evidence)")
plt.show()

# CalibrationDisplay is descriptive here; no recalibration is fit on validation.
assert len(model_validation_probability) == len(validation_data)


## 11. Freeze the design, then open the test envelope once

After selecting the six-feature regularized pipeline, all development rows may
estimate its final scaler and coefficients. We construct a **fresh** pipeline
so no validation-fitted state leaks forward. The test report is evidence for
this exact query, split seed, feature list, target encoding, estimator settings,
and software environment.

In [ ]:
# Combine training and validation only after the modeling contract is frozen.
development_data = pd.concat([train_data, validation_data]).sort_index()
final_pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=1.0, solver="lbfgs", max_iter=2_000, tol=1e-10),
)
final_pipeline.fit(
    development_data[feature_names], development_data["malignant_event"]
)

# Open the sealed test partition once for the predeclared evidence summary.
test_probability = final_pipeline.predict_proba(test_data[feature_names])[:, 1]
test_evidence = evaluate_binary_probabilities(
    test_data["malignant_event"], test_probability
)

assert len(test_probability) == len(test_data)
assert np.all((test_probability >= 0.0) & (test_probability <= 1.0))
print("Final held-out evidence for the instructional pipeline:\n", test_evidence)


## Common failure modes

| Failure | Why it is wrong | Diagnostic |
|---|---|---|
| calling the logit a probability | $z$ is unbounded | check the documented output domain |
| assuming sigmoid output is calibrated | range is not empirical calibration | reliability evidence on the target population |
| summing probabilities instead of multiplying likelihoods | it is a different objective | derive the Bernoulli likelihood first |
| computing `log(sigmoid(z))` naively | extreme scores can overflow or produce `log(0)` | test logits such as $\pm1000$ |
| scaling before splitting | validation/test information enters training | inspect fitted scaler means |
| using `predict()` when probability quality matters | class labels discard ranking and confidence | use `predict_proba()` and proper scores |
| treating $0.5$ as a universal action threshold | costs and capacity are unstated | version a separate policy |
| ignoring perfect separation | coefficients can diverge | monitor norms and use justified regularization |
| interpreting coefficients causally | prediction adjustment is not intervention | state the causal assumptions and design |


In [ ]:
# Stable log loss remains finite for scores that break naive exp/log formulas.
extreme_logits = np.array([-1000.0, 1000.0])
extreme_targets = np.array([1.0, 0.0])
stable_extreme_loss = binary_log_loss_from_logits(
    extreme_logits, extreme_targets
)
stable_extreme_probabilities = sigmoid(extreme_logits)

# The exact probabilities round to 0 and 1 in floating-point arithmetic, but
# the logits-based loss still records a finite, very large penalty.
assert np.isfinite(stable_extreme_loss)
assert stable_extreme_loss == 1000.0
assert np.array_equal(stable_extreme_probabilities, np.array([0.0, 1.0]))
print(f"stable mean loss for two confidently wrong examples: {stable_extreme_loss:.1f}")


## Debugging logistic regression

Use this order; do not tune the learning rate first.

1. **Meaning:** write the observational unit and the event encoded as one.
2. **Shapes:** verify $X\in\mathbb R^{n\times d}$, $y\in\{0,1\}^n$, and parameter alignment.
3. **Evidence boundary:** confirm scaler and coefficients used training rows only.
4. **Forward pass:** inspect logits separately from probabilities.
5. **Loss:** compare the logits formula with a trusted `log_loss` calculation.
6. **Gradient:** run finite differences on a tiny asymmetric example.
7. **Optimization:** plot loss, gradient norm, parameter norm, and termination reason.
8. **Reference:** compare with a library estimator using the *same* regularization contract.
9. **Generalization:** compare against a prevalence baseline on untouched rows.
10. **Decision layer:** inspect thresholded errors only after probability behavior.

If two implementations disagree, first compare target encoding, intercept
handling, scaling, regularization, and class order. “Both are logistic
regression” does not mean they solve identical objectives.


## Professional practice

| Data scientist asks | Software engineer asks |
|---|---|
| What population and event does $p$ describe? | Where are feature order and positive-class semantics validated? |
| Is the Bernoulli/logit specification adequate? | Is stable loss behavior tested at extreme logits? |
| Were model choices made without test feedback? | Are split identities and seeds reproducible? |
| Does probability quality beat a prevalence baseline? | Does the artifact store scaler and model together? |
| Is calibration stable across relevant groups and time? | Are model and decision-policy versions separate? |
| What harms follow false positives or false negatives? | What monitoring joins predictions to delayed outcomes? |

A notebook is the place to derive, visualize, and challenge this model. A
durable implementation should expose typed `fit`, `predict_logit`, and
`predict_proba` contracts; validate schema and class order; persist feature and
model metadata; and test mathematical invariants in CI.


## Guided practice: compute one update by hand

Use two observations with one intercept and one feature:

$$
\widetilde X=\begin{bmatrix}1&-1\\1&2\end{bmatrix},
\quad
\boldsymbol y=\begin{bmatrix}0\\1\end{bmatrix},
\quad
\boldsymbol\theta_0=\begin{bmatrix}0\\0\end{bmatrix}.
$$

1. Show that both initial probabilities equal $1/2$.
2. Compute $\widetilde X^T(\boldsymbol p-\boldsymbol y)/2$.
3. Predict the sign of the new slope after one update with $\eta=0.4$.
4. Run the cell and reconcile every number with your derivation.


In [ ]:
guided_design = np.array([[1.0, -1.0], [1.0, 2.0]])
guided_target = np.array([0.0, 1.0])
guided_parameters = np.zeros(2)
guided_loss, guided_gradient = logistic_loss_and_gradient(
    guided_design, guided_target, guided_parameters
)
guided_updated_parameters = guided_parameters - 0.4 * guided_gradient

assert np.allclose(sigmoid(guided_design @ guided_parameters), 0.5)
assert np.allclose(guided_gradient, np.array([0.0, -0.75]))
assert np.allclose(guided_updated_parameters, np.array([0.0, 0.3]))
print("initial loss:", guided_loss)
print("gradient:", guided_gradient)
print("updated [bias, slope]:", guided_updated_parameters)


## Independent practice: change the scientific representation

Repeat the transparent one-feature workflow with `worst_radius`.

Success criteria:

- fit the scaler on training rows only;
- construct an explicit intercept column;
- verify the analytic gradient at a nonzero parameter vector;
- report training termination and validation log loss;
- compare probabilities with an unregularized scikit-learn reference; and
- explain why a lower validation log loss would not establish causal or clinical superiority.

Then open an issue using the identifier **CLF-PR1** proposing one reusable
component (`stable_sigmoid`, a validated logistic objective, or a fit-result
record). State its public contract and tests before writing package code.


In [ ]:
# A small executable starting point; replace the feature in your own branch.
practice_feature = "worst_radius"
# Learn centering and scale from training rows only.
practice_scaler = StandardScaler().fit(train_data[[practice_feature]])
practice_scaled = practice_scaler.transform(train_data[[practice_feature]])
# Align the explicit intercept column with the derivation.
practice_design = np.column_stack(
    [np.ones(len(practice_scaled)), practice_scaled[:, 0]]
)
practice_fit = fit_logistic_neuron(
    practice_design,
    train_target,
    learning_rate=0.2,
    max_iterations=12_000,
    tolerance=1e-6,
)

# Verify termination and descent before interpreting fitted parameters.
assert practice_fit.converged
assert practice_fit.loss_history[-1] < practice_fit.loss_history[0]
print(
    f"{practice_feature}: {practice_fit.iterations:,} updates; "
    f"final training loss={practice_fit.loss_history[-1]:.4f}"
)


## Extension: regularization, mini-batches, and separation

Choose one extension and make its contract explicit.

1. Add $\lambda\lVert\boldsymbol w\rVert_2^2/2$ without penalizing the bias.
   Derive the new gradient and state whether the Hessian becomes positive definite.
2. Replace the batch gradient with reproducible mini-batches. Compare updates,
   examples processed, wall time, and validation log loss—not epochs alone.
3. Construct a separable two-dimensional training set. Show parameter-norm
   growth for the unregularized fit, then show how L2 regularization changes the result.
4. Implement class weights from a declared cost model. Explain why class
   weighting changes the fitted probability interpretation and may require calibration.

For contribution problem **CLF-PR2**, propose a tested regularized logistic
objective or benchmark recorder. Include finite-difference tests and at least
one failure-mode test.


## Retrieval practice

Answer without executing code.

1. What are the domains of a logit and a sigmoid output?
2. Why is the likelihood a product before taking logarithms?
3. Where does the factor $p(1-p)$ disappear in the cross-entropy gradient?
4. Write the batch gradients with respect to $\boldsymbol w$ and $b$.
5. Which mathematical structure is shared with linear regression?
6. Name three structures that are different from linear regression.
7. Why can separable data prevent a finite unregularized estimate?
8. Why is `C=np.inf` important in the reference comparison but questionable as a default?
9. What does a calibration plot test that ROC AUC does not?
10. Why must a threshold be versioned separately from model parameters?
11. Which denominators define sensitivity, specificity, precision, and NPV?
12. Why can precision change when sensitivity and specificity remain fixed?
13. What population, uncertainty, and subgroup information belongs in a professional report?

## Takeaway

Logistic regression is a single affine neuron with a sigmoid response and a
Bernoulli likelihood. Its gradient

$$
\widetilde X^T(\boldsymbol p-\boldsymbol y)/n
$$

resembles the linear-neuron gradient because both back-propagate an output
error through the same affine score. The response model, objective, curvature,
and interpretation are different. A correct implementation therefore requires
more than swapping an activation: it requires stable loss evaluation, verified
derivatives, explicit target semantics, preserved evidence boundaries, and a
separate decision policy.

The next notebook keeps the affine score $z=\boldsymbol w^T\boldsymbol x+b$
but replaces probability semantics and Bernoulli log loss with signed margins
and hinge loss. That linear-SVM comparison makes the shared computation and
different statistical contracts visible. Only after studying both models will
we design a reusable learning API; thresholds, calibration, asymmetric costs,
and capacity remain a later decision-systems lesson.


## Further reading

- [Python `sqlite3` documentation](https://docs.python.org/3/library/sqlite3.html)
- [NumPy `logaddexp`](https://numpy.org/doc/stable/reference/generated/numpy.logaddexp.html)
- [scikit-learn `LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
- [scikit-learn log-loss documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.log_loss.html)
- [Wisconsin Diagnostic Breast Cancer dataset documentation](https://scikit-learn.org/stable/datasets/toy_dataset.html#breast-cancer-wisconsin-diagnostic-dataset)
- [Wolberg et al., UCI Diagnostic Wisconsin Breast Cancer](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic)
- [Berkson (1944), *Application of the Logistic Function to Bio-Assay*](https://doi.org/10.1080/01621459.1944.10500699)
- [Cox (1958), *The Regression Analysis of Binary Sequences*](https://doi.org/10.1111/j.2517-6161.1958.tb00292.x)

Read documentation as an interface contract. Check the installed version before
assuming defaults—especially regularization, solver, and class ordering.
